In [ ]:
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer

from sklearn.metrics import accuracy_score,f1_score
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn

d:\PPTI 15 ARTEMIS\Semester 8 (Skripsi)\TAM UTAUT\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import re
import emoji

slang_dict = {
    "ga": "tidak",
    "gak": "tidak",
    "gk": "tidak",
    "nggak": "tidak",
    "tp": "tapi",
    "jd": "jadi",
    "bgt": "banget",
    "yg": "yang",
    "krn": "karena"
}

def normalize_slang(text):
    words = text.split()
    return " ".join([slang_dict.get(w, w) for w in words])


def preprocess_text(text):
    text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URL
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # 3. Remove mention
    text = re.sub(r'@\w+', ' ', text)

    # 4. Remove hashtag symbol only (kata tetap)
    text = re.sub(r'#', '', text)

    # 5. Remove emoji
    text = emoji.replace_emoji(text, replace='')

    # 6. Slang normalization ringan
    text = normalize_slang(text)

    # 7. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [3]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

In [4]:
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

In [5]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [6]:
print(train_dataset.column_names)
print(train_dataset[0])

['comment', 'label']
{'comment': 'berbahaya??', 'label': 0}


In [7]:
model_name = "cahya/bert-base-indonesian-1.5G"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    use_safetensors=True
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 514.19it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: cahya/bert-base-indonesian-1.5G
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing f

In [8]:
def tokenize(batch):
    texts = batch["comment"]
    
    # pastikan semua isi list adalah string
    texts = [str(t) for t in texts]

    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [9]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1973 [00:00<?, ? examples/s]

Map: 100%|██████████| 494/494 [00:00<00:00, 12347.58 examples/s]


In [10]:
def compute_metrics(eval_pred):

    logits,labels = eval_pred

    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels,preds)

    f1 = f1_score(labels,preds,average="macro")

    return {"accuracy":acc,"f1":f1}

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="models/sentiment_indoroberta",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",   # ganti dari evaluation_strategy
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01
)

In [12]:
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=test_dataset,
#     processing_class=tokenizer,
#     compute_metrics=compute_metrics
# )

In [13]:
labels = train_df["label"].values

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class Weights:", class_weights)

NameError: name 'compute_class_weight' is not defined

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
class_weights = class_weights.to(device)

In [ ]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,  # ✅ benar
    compute_metrics=compute_metrics
)

In [ ]:
print(train_dataset.column_names)
print(train_dataset[0])

['comment', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
{'comment': 'berbahaya??', 'label': 0, 'input_ids': [3, 8007, 32, 32, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.673719,0.531929,0.771255,0.734651
2,0.389156,0.504073,0.801619,0.768702
3,0.228956,0.539884,0.815789,0.785680


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]


TrainOutput(global_step=372, training_loss=0.4306099030279344, metrics={'train_runtime': 145.4666, 'train_samples_per_second': 40.69, 'train_steps_per_second': 2.557, 'total_flos': 389342079883008.0, 'train_loss': 0.4306099030279344, 'epoch': 3.0})

In [ ]:
trainer.save_model("models/sentiment_indoroberta")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]
